# Scraping Data Dosen PDDIKTI untuk Knowledge Graph

## Overview
Notebook ini digunakan untuk melakukan scraping data dosen dari **PDDIKTI (Pangkalan Data Pendidikan Tinggi)** menggunakan library `pddiktipy`. Data yang dikumpulkan akan digunakan untuk membangun knowledge graph komprehensif tentang dosen dan aktivitas akademik mereka.

## Library Credits & Attribution
Notebook ini menggunakan library `pddiktipy` yang dikembangkan oleh:
- **Author**: [IlhamriSKY](https://github.com/IlhamriSKY)
- **Repository**: [PDDIKTI-kemdikbud-API](https://github.com/IlhamriSKY/PDDIKTI-kemdikbud-API)
- **Description**: Python wrapper untuk mengakses API PDDIKTI Kemdikbud
- **License**: Lihat repository untuk detail lisensi

🙏 **Terima kasih kepada IlhamriSKY** atas kontribusinya dalam menyediakan library yang memudahkan akses data PDDIKTI!

## Data yang di-scrape:
1. **Profil Dosen** - Informasi dasar, jabatan akademik, pendidikan, status kerja
2. **Riwayat Studi** - Pendidikan formal dosen dari S1 hingga S3
3. **Penelitian** - Daftar penelitian yang pernah dilakukan
4. **Pengabdian Masyarakat** - Kegiatan pengabdian kepada masyarakat  
5. **Karya Ilmiah** - Publikasi, jurnal, dan karya tulis ilmiah
6. **Paten** - Hak kekayaan intelektual yang dimiliki
7. **Riwayat Mengajar** - Mata kuliah yang diampu per semester

## Output Files:
Semua data disimpan dalam format CSV di folder `file_tabulars/`:
- `dosen.csv` - Data profil lengkap dosen + ringkasan statistik
- `studi.csv` - Riwayat pendidikan formal
- `penelitian.csv` - Data penelitian
- `pengabdian.csv` - Data pengabdian masyarakat
- `karya.csv` - Data karya ilmiah dan publikasi
- `paten.csv` - Data paten dan HKI
- `mengajar.csv` - Riwayat mengajar per semester

## Konfigurasi Program Studi:
Target program studi dikonfigurasi melalui file eksternal `program_studi_config.txt` untuk memudahkan customization tanpa edit notebook.

---

In [ ]:
# Standard Library Imports
from pathlib import Path
import pandas as pd
from pprint import pprint

# PDDIKTI API Library
from pddiktipy import api

print("📦 Libraries imported successfully!")
print("🔧 Ready to scrape PDDIKTI data")

## 1. Setup Environment dan Dependencies

### Import Libraries
Import semua library yang diperlukan untuk scraping dan pemrosesan data:

In [ ]:
# Setup direktori untuk menyimpan hasil scraping
ROOT_DIR = Path().cwd()
SAVE_DIR_TABULARS = ROOT_DIR / "file_tabulars"

# Setup file konfigurasi program studi
CONFIG_FILE = ROOT_DIR / "program_studi_config.txt"

# Buat direktori jika belum ada
SAVE_DIR_TABULARS.mkdir(parents=True, exist_ok=True)

print(f"📂 Output directory: {SAVE_DIR_TABULARS}")
print(f"⚙️  Config file: {CONFIG_FILE}")
print("✅ Directory setup completed!")

Folder penyimpanan file tabulars: c:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars


### Setup Direktori Output
Membuat direktori untuk menyimpan file CSV hasil scraping:

In [ ]:
def test_api_connection():
    """
    Test koneksi ke PDDIKTI API dan tampilkan struktur data
    """
    keyword = 'Sains Data Universitas Negeri Surabaya'
    
    print("🔄 Testing PDDIKTI API connection...")
    print(f"🔍 Search keyword: {keyword}")
    print("=" * 50)
    
    try:
        with api() as client:
            hasil = client.search_all(keyword)
            
            if hasil:
                print("✅ API connection successful!")
                print(f"📊 Available data categories:")
                
                for category, items in hasil.items():
                    print(f"   📂 {category}: {len(items)} items")
                
                # Tampilkan sample data dosen jika ada
                dosen_data = hasil.get("dosen", [])
                if dosen_data:
                    print(f"\\n🔍 Sample dosen data (first item):")
                    sample_dosen = dosen_data[0]
                    for key, value in sample_dosen.items():
                        print(f"   {key}: {value}")
                        
                return hasil
            else:
                print("⚠️ No data found for the given keyword")
                return None
                
    except Exception as e:
        print(f"❌ API connection failed: {e}")
        return None

# Jalankan test
test_result = test_api_connection()

INFO:pddiktipy.api:PDDIKTI API client initialized successfully


{'mahasiswa': [{'id': '05h_Q4hsbv7NOOLrxuK-21JANWsDtRIC9_KPLLOgo8tSHuUh5Xlo5C2_KeP08stKITsbKA==', 'nama': 'MUHAMMAD DAFA ALVIAN RAMADHANI', 'nim': '23031554017', 'nama_pt': 'UNIVERSITAS NEGERI SURABAYA', 'sinkatan_pt': 'UNESA', 'nama_prodi': 'SAINS DATA'}, {'id': 'mxVEZRu7SEfSGIolpfRI_FaRBjDGrtATni1PnUd5QG5mzcG1eiWT12V065zQwTom8Aigug==', 'nama': 'NOVITASARI', 'nim': '23031554194', 'nama_pt': 'UNIVERSITAS NEGERI SURABAYA', 'sinkatan_pt': 'UNESA', 'nama_prodi': 'SAINS DATA'}, {'id': 'jBbloWXh1v9sRIbuifP5VkSXZbouIDWKzK2MEWcXsD14JOw-cnSUnepc1gPNPc3dfQM92w==', 'nama': 'AMILIYA', 'nim': '25031554163', 'nama_pt': 'UNIVERSITAS NEGERI SURABAYA', 'sinkatan_pt': 'UNESA', 'nama_prodi': 'SAINS DATA'}, {'id': 'ozgeUgxxTDH-DcNhttiRPHifXM_f6ST8y0X_nJ0tasBm8y4s1RrmODoBbaSHdpZXkzGDMQ==', 'nama': 'ANALICIA', 'nim': '22031554007', 'nama_pt': 'UNIVERSITAS NEGERI SURABAYA', 'sinkatan_pt': 'UNESA', 'nama_prodi': 'SAINS DATA'}, {'id': 'S6QlokapdBd4wOg_olo105YdqKIKukN8KrLco2r4Ah08EWbUqLBYOhI-tG5J-2ZA3ym2BA==',

## 2. Testing PDDIKTI API Connection

### Test Koneksi Dasar
Uji koneksi ke PDDIKTI API dan lihat struktur data yang tersedia:

In [ ]:
def explore_api_methods():
    """
    Eksplorasi berbagai method API untuk mengambil detail data dosen
    """
    keyword = "Sains Data Universitas Negeri Surabaya"
    
    print("🔬 Exploring PDDIKTI API methods...")
    print("=" * 50)
    
    try:
        with api() as client:
            # Search dosen
            hasil = client.search_all(keyword) or {}
            dosen_list = hasil.get("dosen", [])
            
            if not dosen_list:
                print("⚠️ No dosen data found")
                return
            
            # Ambil sample dosen pertama
            sample_dosen = dosen_list[0]
            dosen_id = sample_dosen.get("id")
            nama_dosen = sample_dosen.get("nama", "Unknown")
            
            print(f"📋 Testing with dosen: {nama_dosen} (ID: {dosen_id})")
            print("=" * 50)
            
            # Test berbagai API methods
            api_methods = [
                ("get_dosen_profile", "👤 Profile"),
                ("get_dosen_study_history", "🎓 Study History"), 
                ("get_dosen_penelitian", "🔬 Penelitian"),
                ("get_dosen_pengabdian", "🤝 Pengabdian"),
                ("get_dosen_karya", "📚 Karya Ilmiah"),
                ("get_dosen_paten", "💡 Paten"),
                ("get_dosen_teaching_history", "🏫 Teaching History")
            ]
            
            for method_name, description in api_methods:
                try:
                    method = getattr(client, method_name)
                    data = method(dosen_id) or []
                    
                    if isinstance(data, list):
                        count = len(data)
                    elif isinstance(data, dict):
                        count = 1 if data else 0
                    else:
                        count = 1 if data else 0
                    
                    print(f"   {description}: {count} records")
                    
                    # Tampilkan sample jika ada data
                    if data and isinstance(data, list) and len(data) > 0:
                        print(f"      Sample fields: {list(data[0].keys())}")
                    elif data and isinstance(data, dict):
                        print(f"      Sample fields: {list(data.keys())}")
                        
                except Exception as e:
                    print(f"   {description}: ❌ Error - {e}")
            
            print("\\n✅ API methods exploration completed!")
            
    except Exception as e:
        print(f"❌ Exploration failed: {e}")

# Jalankan eksplorasi (uncomment untuk test)
# explore_api_methods()

INFO:pddiktipy.api:PDDIKTI API client initialized successfully



=== Fields untuk 'dosen' di Prodi: Sains Data ===
- {'id': 'GjMJGeaQwpNAmFEJhwT-n96KkDasFXAu3Y85MukYePT0gqG41laY6iaSZBufkk7EcC2gWQ==', 'nama': 'ATIK WINTARTI', 'nidn': '0012106608', 'nuptk': '4344744645230103', 'nama_pt': 'UNIVERSITAS NEGERI SURABAYA', 'sinkatan_pt': 'UNESA', 'nama_prodi': 'SAINS DATA'}
- {'id': 'e6z33N1ci_aYbZHstGko44DSrCL1xC7tzUP2q8oOcdCdimaX6EZRJnYdtED9tgv-aZoP0w==', 'nama': 'ULFA SITI NURAINI', 'nidn': '0004109601', 'nuptk': '5336774675230183', 'nama_pt': 'UNIVERSITAS NEGERI SURABAYA', 'sinkatan_pt': 'UNESA', 'nama_prodi': 'SAINS DATA'}
- {'id': '3pwujcqBE4kCIx6FbFdYknCDi8mnkQdqU_ejHw9o4DgCcebVYoC47IzUyh2sIm8l1Me0tQ==', 'nama': 'HASANUDDIN AL-HABIB', 'nidn': '0009089303', 'nuptk': '3141771672130313', 'nama_pt': 'UNIVERSITAS NEGERI SURABAYA', 'sinkatan_pt': 'UNESA', 'nama_prodi': 'SAINS DATA'}
- {'id': 'tbkWDd-rJmozd7f00FN8IGCQVM3oZ_crY7R95gSx_B7rdK-jkWXMKgE3OURykXotgDOOLA==', 'nama': 'YUNI ROSITA DEWI', 'nidn': '0728069501', 'nuptk': '2960773674230242', 'nama_pt':

### Eksplorasi Detail API Methods
Test berbagai method yang tersedia untuk mengambil data detail dosen:

## 3. Fungsi Utility dan Helper

### Utility Functions
Fungsi-fungsi bantuan untuk pemrosesan data:

In [ ]:
def title_case(text: str) -> str:
    """
    Convert text to title case (capitalize each word)
    
    Args:
        text (str): Input text to convert
        
    Returns:
        str: Text in title case format
    """
    if not text or not isinstance(text, str):
        return ""
    return " ".join(word.capitalize() for word in text.split())

def safe_get_nidn(study_list: list, profile_data: dict) -> str:
    """
    Safely extract NIDN from study history or profile data
    
    Args:
        study_list (list): List of study history records
        profile_data (dict): Profile data from API
        
    Returns:
        str: NIDN if found, empty string otherwise
    """
    # Try to get NIDN from study history first
    for record in study_list:
        nidn = record.get("nidn")
        if nidn and nidn.strip():
            return nidn.strip()
    
    # Fallback to profile data
    return profile_data.get("nidn", "").strip()

def print_progress(current: int, total: int, dosen_name: str):
    """
    Print progress information for scraping process
    
    Args:
        current (int): Current dosen index
        total (int): Total number of dosen
        dosen_name (str): Name of current dosen being processed
    """
    percentage = (current / total) * 100
    print(f"[{current:3d}/{total}] ({percentage:5.1f}%) Processing: {dosen_name}")

def load_program_studi_config(config_file_path: Path) -> list:
    """
    Load program studi configuration from external text file
    
    Args:
        config_file_path (Path): Path to configuration file
        
    Returns:
        list: List of program studi names to scrape
        
    Raises:
        FileNotFoundError: If config file doesn't exist
        ValueError: If no valid program studi found in config
    """
    if not config_file_path.exists():
        raise FileNotFoundError(
            f"Configuration file not found: {config_file_path}\n"
            f"Please create the file with program studi names (one per line)"
        )
    
    program_studi_list = []
    
    print(f"📖 Reading configuration from: {config_file_path.name}")
    
    try:
        with open(config_file_path, 'r', encoding='utf-8') as file:
            for line_num, line in enumerate(file, 1):
                line = line.strip()
                
                # Skip empty lines and comments (lines starting with #)
                if not line or line.startswith('#'):
                    continue
                
                # Add to list if it's a valid program studi name
                if len(line) >= 3:  # Minimum reasonable length
                    program_studi_list.append(line)
                    print(f"   ✅ Line {line_num}: {line}")
                else:
                    print(f"   ⚠️  Line {line_num}: '{line}' - Skipped (too short)")
    
    except Exception as e:
        raise ValueError(f"Error reading config file: {e}")
    
    if not program_studi_list:
        raise ValueError(
            f"No valid program studi found in {config_file_path.name}\n"
            f"Please add program studi names (uncommented lines) to the file"
        )
    
    print(f"🎯 Total program studi loaded: {len(program_studi_list)}")
    return program_studi_list

## 4. Data Collection Functions

### Dosen Search and Collection
Fungsi untuk mencari dan mengumpulkan data dosen berdasarkan program studi:

In [ ]:
def collect_unique_dosen(prodi_keywords: list, client) -> dict:
    """
    Collect unique dosen from multiple program studi
    
    Args:
        prodi_keywords (list): List of program studi keywords to search
        client: PDDIKTI API client instance
        
    Returns:
        dict: Dictionary with dosen_id as key and dosen data as value
    """
    print("🔍 Collecting unique dosen from program studi...")
    dosen_set = {}
    
    for prodi in prodi_keywords:
        print(f"   📚 Searching: {prodi} UNESA")
        
        try:
            hasil_all = client.search_all(f"{prodi} UNESA") or {}
            dosen_list = hasil_all.get("dosen", [])
            
            # Filter dosen yang benar-benar dari prodi yang dimaksud
            for d in dosen_list:
                nama_prodi = d.get("nama_prodi", "").lower()
                if prodi.lower() in nama_prodi:
                    dosen_id = d.get("id")
                    if dosen_id:
                        dosen_set[dosen_id] = d
                        
            print(f"      ✅ Found {len([d for d in dosen_list if prodi.lower() in d.get('nama_prodi', '').lower()])} dosen")
            
        except Exception as e:
            print(f"      ❌ Error searching {prodi}: {e}")
    
    print(f"🎯 Total unique dosen collected: {len(dosen_set)}")
    return dosen_set

### Individual Dosen Data Extraction
Fungsi untuk mengambil semua data detail dari seorang dosen:

In [ ]:
def extract_dosen_details(dosen_id: str, dosen_data: dict, client) -> dict:
    """
    Extract comprehensive details for a single dosen
    
    Args:
        dosen_id (str): Unique identifier for dosen
        dosen_data (dict): Basic dosen data from search
        client: PDDIKTI API client instance
        
    Returns:
        dict: Dictionary containing all data lists for the dosen
    """
    nama_dosen = title_case(dosen_data.get("nama", "").lower())
    
    try:
        # Fetch all data types using API
        data_collections = {
            'study_list': client.get_dosen_study_history(dosen_id) or [],
            'penelitian_list': client.get_dosen_penelitian(dosen_id) or [],
            'pengabdian_list': client.get_dosen_pengabdian(dosen_id) or [],
            'karya_list': client.get_dosen_karya(dosen_id) or [],
            'paten_list': client.get_dosen_paten(dosen_id) or [],
            'teaching_list': client.get_dosen_teaching_history(dosen_id) or [],
            'profile_data': client.get_dosen_profile(dosen_id) or {}
        }
        
        return data_collections
        
    except Exception as e:
        print(f"      ❌ Error extracting details for {nama_dosen}: {e}")
        return {
            'study_list': [],
            'penelitian_list': [],
            'pengabdian_list': [],
            'karya_list': [],
            'paten_list': [],
            'teaching_list': [],
            'profile_data': {}
        }

### Data Processing Functions
Fungsi untuk memproses dan membuat record data untuk setiap kategori:

In [ ]:
def create_dosen_record(dosen_id: str, dosen_data: dict, details: dict) -> dict:
    """
    Create comprehensive dosen record with statistics
    
    Args:
        dosen_id (str): Dosen unique identifier
        dosen_data (dict): Basic dosen information
        details (dict): All detailed data collections
        
    Returns:
        dict: Complete dosen record with statistics
    """
    nama_dosen = title_case(dosen_data.get("nama", "").lower())
    profile_data = details['profile_data']
    study_list = details['study_list']
    
    # Get final NIDN
    nidn_final = safe_get_nidn(study_list, profile_data)
    
    return {
        "id_sdm": dosen_id,
        "nama_dosen": nama_dosen,
        "nidn": nidn_final,
        "jabatan_akademik": profile_data.get("jabatan_akademik", ""),
        "pendidikan_tertinggi": profile_data.get("pendidikan_tertinggi", ""),
        "status_ikatan_kerja": profile_data.get("status_ikatan_kerja", ""),
        "status_aktivitas": profile_data.get("status_aktivitas", ""),
        "nama_pt": dosen_data.get("nama_pt", ""),
        "nama_prodi": dosen_data.get("nama_prodi", ""),
        "jumlah_penelitian": len(details['penelitian_list']),
        "jumlah_pengabdian": len(details['pengabdian_list']),
        "jumlah_karya_ilmiah": len(details['karya_list']),
        "jumlah_paten": len(details['paten_list'])
    }

def create_activity_records(dosen_id: str, nama_dosen: str, activity_list: list, activity_type: str) -> list:
    """
    Create records for activities (penelitian, pengabdian, karya, paten)
    
    Args:
        dosen_id (str): Dosen unique identifier
        nama_dosen (str): Dosen name in title case
        activity_list (list): List of activities
        activity_type (str): Type of activity for logging
        
    Returns:
        list: List of activity records
    """
    records = []
    for record in activity_list:
        records.append({
            "id_sdm": dosen_id,
            "nama_dosen": nama_dosen,
            "judul_kegiatan": record.get("judul_kegiatan", ""),
            "tahun_kegiatan": record.get("tahun_kegiatan", ""),
            "jenis_kegiatan": record.get("jenis_kegiatan", "")
        })
    return records

def create_study_records(dosen_id: str, nama_dosen: str, study_list: list) -> list:
    """
    Create study history records
    
    Args:
        dosen_id (str): Dosen unique identifier  
        nama_dosen (str): Dosen name in title case
        study_list (list): List of study history
        
    Returns:
        list: List of study records
    """
    records = []
    for record in study_list:
        records.append({
            "id_sdm": dosen_id,
            "nama_dosen": nama_dosen,
            "nidn": record.get("nidn", ""),
            "jenjang": record.get("jenjang", ""),
            "nama_pt": record.get("nama_pt", ""),
            "nama_prodi": record.get("nama_prodi", ""),
            "tahun_masuk": record.get("tahun_masuk", ""),
            "tahun_lulus": record.get("tahun_lulus", ""),
            "gelar_akademik": record.get("gelar_akademik", ""),
            "singkatan_gelar": record.get("singkatan_gelar", "")
        })
    return records

def create_teaching_records(dosen_id: str, nama_dosen: str, teaching_list: list) -> list:
    """
    Create teaching history records
    
    Args:
        dosen_id (str): Dosen unique identifier
        nama_dosen (str): Dosen name in title case  
        teaching_list (list): List of teaching history
        
    Returns:
        list: List of teaching records
    """
    records = []
    for record in teaching_list:
        records.append({
            "id_sdm": dosen_id,
            "nama_dosen": nama_dosen,
            "nama_semester": record.get("nama_semester", ""),
            "kode_matkul": record.get("kode_matkul", ""),
            "nama_matkul": title_case(record.get("nama_matkul", "").lower()),
            "nama_kelas": record.get("nama_kelas", ""),
            "nama_pt": record.get("nama_pt", "")
        })
    return records

## 5. Main Scraping Function

### Fungsi Utama yang Modular
Fungsi utama untuk mengumpulkan semua data dosen yang telah direfactor menjadi lebih modular:

In [ ]:
def collect_dosen_knowledge_graph(prodi_keywords: list) -> None:
    """
    Main function to collect comprehensive dosen data for building knowledge graph
    
    Args:
        prodi_keywords (list): List of program studi keywords to search
        
    Process:
        1. Collect unique dosen from all program studi
        2. Extract detailed data for each dosen
        3. Process and organize data into structured records
        4. Save all data to CSV files
    """
    print("🚀 STARTING COMPREHENSIVE DOSEN DATA COLLECTION")
    print("=" * 60)
    
    # Initialize data containers
    all_data = {
        "dosen": [],
        "studi": [],
        "penelitian": [],
        "pengabdian": [],
        "karya": [],
        "paten": [],
        "mengajar": []
    }
    
    try:
        with api() as client:
            # Phase 1: Collect unique dosen
            print("\\n📋 PHASE 1: Collecting unique dosen...")
            dosen_set = collect_unique_dosen(prodi_keywords, client)
            
            if not dosen_set:
                print("❌ No dosen found for the specified program studi")
                return
            
            # Phase 2: Extract detailed data for each dosen  
            print(f"\\n🔍 PHASE 2: Extracting detailed data for {len(dosen_set)} dosen...")
            print("-" * 60)
            
            for index, (dosen_id, dosen_data) in enumerate(dosen_set.items(), 1):
                nama_dosen = title_case(dosen_data.get("nama", "").lower())
                print_progress(index, len(dosen_set), nama_dosen)
                
                # Extract comprehensive details
                details = extract_dosen_details(dosen_id, dosen_data, client)
                
                # Create dosen profile record
                dosen_record = create_dosen_record(dosen_id, dosen_data, details)
                all_data["dosen"].append(dosen_record)
                
                # Create related records
                all_data["studi"].extend(
                    create_study_records(dosen_id, nama_dosen, details['study_list'])
                )
                all_data["penelitian"].extend(
                    create_activity_records(dosen_id, nama_dosen, details['penelitian_list'], "penelitian")
                )
                all_data["pengabdian"].extend(
                    create_activity_records(dosen_id, nama_dosen, details['pengabdian_list'], "pengabdian")
                )
                all_data["karya"].extend(
                    create_activity_records(dosen_id, nama_dosen, details['karya_list'], "karya")
                )
                all_data["paten"].extend(
                    create_activity_records(dosen_id, nama_dosen, details['paten_list'], "paten")
                )
                all_data["mengajar"].extend(
                    create_teaching_records(dosen_id, nama_dosen, details['teaching_list'])
                )
            
            # Phase 3: Save all data to CSV files
            print(f"\\n💾 PHASE 3: Saving data to CSV files...")
            print("-" * 60)
            
            for data_type, records in all_data.items():
                if records:  # Only save if there's data
                    df = pd.DataFrame(records)
                    output_file = SAVE_DIR_TABULARS / f"{data_type}.csv"
                    df.to_csv(output_file, index=False)
                    print(f"   ✅ {data_type.capitalize()}: {len(records)} records → {output_file.name}")
                else:
                    print(f"   ⚠️  {data_type.capitalize()}: No data to save")
            
            # Final summary
            print(f"\\n" + "=" * 60)
            print("🎯 DATA COLLECTION COMPLETED!")
            print("=" * 60)
            print(f"📊 Final Statistics:")
            for data_type, records in all_data.items():
                print(f"   📋 {data_type.capitalize()}: {len(records)} records")
            
            print(f"\\n📂 All files saved in: {SAVE_DIR_TABULARS}")
            print("✅ Knowledge graph data collection successful!")
            
    except Exception as e:
        print(f"\\n❌ Data collection failed: {e}")
        raise

## 6. Eksekusi dan Contoh Penggunaan

### Menjalankan Scraping
Untuk menjalankan scraping data dosen, jalankan cell di bawah ini:

In [ ]:
def main():
    """
    Main execution function with program studi loaded from config file
    """
    print("🚀 STARTING PDDIKTI DOSEN DATA COLLECTION")
    print("=" * 60)
    
    try:
        # Load program studi dari file konfigurasi
        print("⚙️  Loading program studi configuration...")
        program_studi_target = load_program_studi_config(CONFIG_FILE)
        
        print(f"\\n🎯 Target Program Studi (dari {CONFIG_FILE.name}):")
        for i, prodi in enumerate(program_studi_target, 1):
            print(f"   {i:2d}. 📚 {prodi}")
        
        print(f"\\n📋 Total program studi yang akan di-scrape: {len(program_studi_target)}")
        
        # Konfirmasi sebelum mulai scraping
        print("\\n" + "⏰" + " Scraping akan dimulai...")
        print("💡 Tip: Edit file 'program_studi_config.txt' untuk mengubah target program studi")
        print("-" * 60)
        
        # Jalankan scraping
        collect_dosen_knowledge_graph(program_studi_target)
        
    except FileNotFoundError as e:
        print(f"❌ Configuration file error: {e}")
        print("\\n🔧 Cara mengatasi:")
        print("   1. Pastikan file 'program_studi_config.txt' ada di direktori yang sama")
        print("   2. File sudah berisi nama program studi (satu per baris)")
        print("   3. Hapus tanda # di depan nama program studi yang ingin diaktifkan")
        
    except ValueError as e:
        print(f"❌ Configuration error: {e}")
        print("\\n🔧 Cara mengatasi:")
        print("   1. Buka file 'program_studi_config.txt'")
        print("   2. Pastikan ada minimal satu program studi yang tidak dikomentari (#)")
        print("   3. Contoh format yang benar:")
        print("      Sains Data")
        print("      Kecerdasan Artifisial")
        
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        print("\\n🔧 Silakan periksa:")
        print("   1. Koneksi internet")
        print("   2. File konfigurasi")
        print("   3. Permissions folder")

# Jalankan scraping (uncomment untuk eksekusi)
if __name__ == "__main__":
    main()